# BigQuery Omniscience Media Archive: Multimodal AI & Vector Search

This notebook guides you through the **BigQuery Omniscience Media Archive** demo. It showcases:
1. **Autonomous Ingestion & Multimodal Embeddings:** Automatically generating 3072-dimensional multimodal embeddings in the background using the native `AI.EMBED` generated column and Vertex AI.
2. **Cross-Modal Semantic Search:** Querying unstructured video files using text descriptions and clipped speech audio snippets.
3. **High-Precision Hybrid Search:** Combining visual location queries (using U.S. Capitol photos) with keyword text filtering to isolate specific speeches.

In [ ]:
# ==========================================
# PARAMETERS CONFIGURATION
# ==========================================
# Replace these values with your actual Google Cloud configurations

PROJECT_ID = "your-project-id"
DATASET_ID = "your_dataset_id"
BUCKET_NAME = "your-bucket-name"
CONNECTION_ID = f"{PROJECT_ID}.us.your-connection-id"  # Cloud Resource Connection Name


In [ ]:
import pandas as pd
from google.cloud import bigquery

# Initialize BigQuery Client
client = bigquery.Client(project=PROJECT_ID)

def run_sql(query_string):
    """
    Helper function to format and run SQL queries with active configuration parameters.
    """
    formatted_query = (
        query_string
        .replace("<PROJECT_ID>", PROJECT_ID)
        .replace("<DATASET_ID>", DATASET_ID)
        .replace("<BUCKET_NAME>", BUCKET_NAME)
        .replace("<CONNECTION_ID>", CONNECTION_ID)
    )
    
    try:
        query_job = client.query(formatted_query)
        df = query_job.to_dataframe()
        return df
    except Exception as e:
        print(f"[ERROR] Query failed: {e}")
        return None


## 🚀 Step 1: Ingest and Automate (Setup)

In this step, we will:
1. Register the Vertex AI multimodal embedding model.
2. Create an **external GCS Object Table** (`assets_metadata`) pointing directly to the Cloud Storage bucket speeches folder. This table dynamically and autonomously discovers all speech files on GCS.
3. Define the standard table `assets` with **Autonomous Embedding** enabled using the native `AI.EMBED` generated column.
4. Autonomously synchronize speech videos from the Object Table into the standard table in a single query.
5. Monitor the asynchronous background embedding generation progress.
6. Flatten the structural embeddings into a flat table (`assets_searchable`) for fast vector searches.

In [ ]:
# 1. Create remote model, external GCS Object Table, and base assets table
setup_sql = """
-- Create remote Vertex AI Multimodal model
CREATE OR REPLACE MODEL `<PROJECT_ID>.<DATASET_ID>.multimodal_model`
REMOTE WITH CONNECTION `<CONNECTION_ID>`
OPTIONS (ENDPOINT = 'gemini-embedding-2-preview');

-- Create external GCS Object Table to dynamically discover speech files in GCS
CREATE OR REPLACE EXTERNAL TABLE `<PROJECT_ID>.<DATASET_ID>.assets_metadata`
WITH CONNECTION `<CONNECTION_ID>`
OPTIONS (
  object_metadata = 'SIMPLE',
  uris = ['gs://<BUCKET_NAME>/media_archive/speeches/*']
);

-- Create table with AI.EMBED generated column
CREATE OR REPLACE TABLE `<PROJECT_ID>.<DATASET_ID>.assets` (
  asset_id STRING,
  uri STRING,
  asset_embedding STRUCT<result ARRAY<FLOAT64>, status STRING>
    GENERATED ALWAYS AS (
      AI.EMBED(
        uri,
        connection_id => '<CONNECTION_ID>',
        endpoint => 'gemini-embedding-2-preview'
      )
    ) STORED OPTIONS (asynchronous = TRUE)
);
"""

print("Setting up BigQuery remote model, external GCS Object Table, and base table...")
run_sql(setup_sql)
print("Done.")


In [ ]:
# 2. Autonomously synchronize new files from GCS Object Table into base assets table
ingestion_sql = """
INSERT INTO `<PROJECT_ID>.<DATASET_ID>.assets` (asset_id, uri)
SELECT
  REGEXP_EXTRACT(uri, r'([^/]+)\.[^.]+$') AS asset_id, -- Extract file name
  uri
FROM
  `<PROJECT_ID>.<DATASET_ID>.assets_metadata`
WHERE
  uri NOT IN (SELECT uri FROM `<PROJECT_ID>.<DATASET_ID>.assets`);
"""

print("Synchronizing GCS files from Object Table to standard assets table...")
run_sql(ingestion_sql)
print("Done. Asynchronous multimodal embedding generation has started autonomously in the background!")


In [ ]:
# 3. Monitor progress and flatten structural embeddings into flat assets_searchable table
monitor_sql = """
SELECT
  COUNT(*) AS total_num_rows,
  COUNTIF(asset_embedding IS NOT NULL AND asset_embedding.status = '') AS total_num_generated_embeddings,
  COUNTIF(asset_embedding IS NOT NULL AND asset_embedding.status != '') AS total_num_failed_embeddings
FROM
  `<PROJECT_ID>.<DATASET_ID>.assets`;
"""

flatten_sql = """
CREATE OR REPLACE TABLE `<PROJECT_ID>.<DATASET_ID>.assets_searchable` AS
SELECT
  asset_id,
  uri,
  asset_embedding.result AS embedding
FROM
  `<PROJECT_ID>.<DATASET_ID>.assets`;
"""

import time
print("Polling background autonomous embedding generation progress...")
for i in range(5):
    df = run_sql(monitor_sql)
    total = df['total_num_rows'].iloc[0]
    synced = df['total_num_generated_embeddings'].iloc[0]
    failed = df['total_num_failed_embeddings'].iloc[0]
    print(f"Attempt {i+1}/5: {synced}/{total} rows generated successfully (Failed: {failed})")
    if synced == total:
        break
    time.sleep(5)

print("\nMaterializing flat assets_searchable table...")
run_sql(flatten_sql)
print("Done. Table 'assets_searchable' is ready for queries!")


## 🔍 Step 2: Cross-Modal Discovery

In this step, we demonstrate how to query unstructured videos using multiple search modalities:
1. **Scene A: Text-to-Video Search** - Find matching speech videos using a natural language description ("historical political speech given at night in a stadium").
2. **Scene B: Audio-to-Video Search** - Locate the full speech video recording using a short, 22-second clipped audio snippet of Martin Luther King Jr.'s *"I have a dream"* speech.

In [ ]:
# 1. Scene A: Text-to-Video search
text_search_sql = """
WITH text_query AS (
  SELECT ml_generate_embedding_result AS vector
  FROM ML.GENERATE_EMBEDDING(
    MODEL `<PROJECT_ID>.<DATASET_ID>.multimodal_model`,
    (SELECT 'historical political speech given at night in a stadium' AS content)
  )
)
SELECT base.asset_id, distance
FROM VECTOR_SEARCH(
  TABLE `<PROJECT_ID>.<DATASET_ID>.assets_searchable`,
  'embedding',
  (SELECT vector FROM text_query),
  top_k => 5,
  distance_type => 'COSINE'
);
"""

print("Running Scene A: Text-to-Video Search...")
df_text = run_sql(text_search_sql)
print("\nSearch Results:")
print(df_text)


In [ ]:
# 2. Scene B: Audio-to-Video search
audio_search_sql = """
WITH audio_query AS (
  SELECT ml_generate_embedding_result AS vector
  FROM ML.GENERATE_EMBEDDING(
    MODEL `<PROJECT_ID>.<DATASET_ID>.multimodal_model`,
    (SELECT 'gs://<BUCKET_NAME>/media_archive/queries/1-MLK-Dream_clipped.mp3' AS content)
  )
)
SELECT base.asset_id, distance
FROM VECTOR_SEARCH(
  TABLE `<PROJECT_ID>.<DATASET_ID>.assets_searchable`,
  'embedding',
  (SELECT vector FROM audio_query),
  top_k => 5,
  distance_type => 'COSINE'
);
"""

print("Running Scene B: Audio-to-Video Search...")
df_audio = run_sql(audio_search_sql)
print("\nSearch Results (Closer to 0 = better match):")
print(df_audio)


## 🎯 Step 3: Precision Retrieval with Multimodal Hybrid Search

This step showcases the ultimate precision capability of BigQuery Vector Search by combining **visual semantic search** and **text keyword search** in a single query:
* **Visual Context:** A photo of the **U.S. Capitol** (`capitol-dc.png`) is passed as the visual query. The multimodal model automatically links this photo to speech videos delivered in front of the U.S. Capitol.
* **Textual Keyword:** The keyword term `'inaugural'` is applied using full-text `SEARCH`.
* **Result:** By unifying visual location context and table metadata, BigQuery instantly pinpoints **JFK's Inaugural Address** (`08_jfk_inaugural`) as the perfect match!

> [!TIP]
> **Advanced Showcase:** Swap the tourism photo (`capitol-dc.png`) for the protest photo (`protest.png`) and filter for the keyword `'rfk'` or `'assassination'` to retrieve **Robert F. Kennedy's Remarks on MLK's Assassination** (`10_rfk_mlk_assassination`), demonstrating how the model understands the mood/context of the images!

In [ ]:
# 1. Step 3: High-precision hybrid search
hybrid_search_sql = """
WITH reference_image_embedding AS (
  SELECT ml_generate_embedding_result AS vector
  FROM ML.GENERATE_EMBEDDING(
    MODEL `<PROJECT_ID>.<DATASET_ID>.multimodal_model`,
    (SELECT 'gs://<BUCKET_NAME>/media_archive/hybrid/capitol-dc.png' AS content)
  )
)
SELECT base.asset_id, distance
FROM VECTOR_SEARCH(
  TABLE `<PROJECT_ID>.<DATASET_ID>.assets_searchable`,
  'embedding',
  (SELECT vector FROM reference_image_embedding),
  top_k => 10
)
WHERE SEARCH(base, 'inaugural');
"""

print("Running Step 3: High-Precision Multimodal Hybrid Search...")
df_hybrid = run_sql(hybrid_search_sql)
print("\nSearch Results:")
print(df_hybrid)
